In [151]:
import numpy as np
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize
import string
import ast
import os
from pathlib import Path

## Data Cleaning

In [ ]:
# Using os.path - go up one directory and access the file
data_path = os.path.join(os.path.dirname(os.path.abspath('')), 'job_descriptions.csv')
df = pd.read_csv(data_path)

In [158]:
# drop the url since it is not a relevant predictor
clean_df = df.drop('url', axis=1)

# split data into jobs and skills
jobs = clean_df['job']                  
skills = clean_df['skills']

### Job Description

We tokenize by word and add a `job_id` column.

In [159]:
# Standardize punctuation to prevent random biases.

punkt = string.punctuation
punkt = punkt[0:2] + punkt[3:13] + punkt[14:-1]

for i in range(len(jobs)):
    jobs.loc[i] = jobs.loc[i].lower()

    punkt_str = jobs.loc[i]
    test_str = punkt_str.translate(str.maketrans('', '', punkt))
    jobs.loc[i] = test_str

In [160]:
for i in range(len(jobs)):
    jobs.loc[i] = word_tokenize(jobs[i])

jobs = jobs.explode(ignore_index=False).reset_index(drop=False)

### Skills

The skills data is in a `string` format, thus we convert from a string to a list using this claude generated function.

In [165]:
# Assuming df is your dataframe and 'skills' is your column name
def convert_string_to_list(skills_str):
    try:
        # For properly formatted strings like "[item1, item2, item3]"
        return ast.literal_eval(skills_str)
    except (ValueError, SyntaxError):
        # Handle potential errors or edge cases
        if isinstance(skills_str, list):
            # If it's already a list, return it as is
            return skills_str
        elif isinstance(skills_str, str):
            # For simple comma-separated strings without brackets
            if skills_str.strip() == '':
                return []
            if '[' not in skills_str and ']' not in skills_str:
                return [item.strip() for item in skills_str.split(',')]
        # Default fallback
        return []

# Apply the conversion to the skills column
skills = skills.apply(convert_string_to_list)